# 07 — Experimental analysis, threshold tuning and ablation
This notebook collects exported result CSVs, calculates a recall-oriented threshold on validation predictions, and produces comparison plots. Never choose the threshold using the final test labels.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, fbeta_score, confusion_matrix, classification_report
RESULTS=Path("../results")
if not RESULTS.exists(): RESULTS=Path("results")
RESULTS.mkdir(exist_ok=True,parents=True)

In [ ]:
# Example threshold-selection helper.
# Supply out-of-fold TRAINING scores and labels, not test scores/labels.
def choose_f2_threshold(y_train, oof_scores, min_recall=0.80):
    precision, recall, thresholds = precision_recall_curve(y_train, oof_scores)
    # precision/recall have one extra terminal element vs thresholds
    p, r = precision[:-1], recall[:-1]
    f2 = (5*p*r)/(4*p+r+1e-12)
    valid = np.where(r >= min_recall)[0]
    if len(valid)==0:
        best=int(np.argmax(f2))
    else:
        best=valid[np.argmax(f2[valid])]
    return float(thresholds[best]), float(p[best]), float(r[best]), float(f2[best])
print("Threshold helper ready. Call it with training-only out-of-fold scores.")

In [ ]:
# Load result tables if they exist
for fname in ["classical_cv.csv","feature_selection_cv.csv","ablation_results.csv"]:
    path=RESULTS/fname
    if path.exists():
        print("\\n",fname)
        display(pd.read_csv(path))
    else:
        print(fname, "not yet generated — complete the corresponding experiment first.")

## Ablation plan
Run controlled comparisons where only one component changes:
1. Median vs KNN imputation, keeping model/features/threshold fixed.
2. No selection vs MI vs PCA vs RFE, keeping imputation/model fixed.
3. Classical RBF-SVM vs QSVM vs VQC on a clearly stated matched sample budget.
4. Default 0.50 threshold vs a threshold selected using training-only out-of-fold predictions.
5. Report runtime, sample count, qubits, circuit depth and number of kernel evaluations alongside predictive metrics.

Use a common held-out test set for final evaluation. Quantum training subsets must be documented; do not compare a 200-row quantum training run as though it used the full training data.